# 09 · Doctor: fair model selection, significance and error analysis

**SimClinic, CS425 Group 7 (doctor model).** Notebook 05 left three questions open. This notebook answers them.

1. **Which fine-tuned doctor should ship?** Notebook 04 chose GPT-2 small over medium on 50 dev cases, by two cases. Medium then did better on test. Picking medium *because* of the test result would mean selecting on the test set. Here every grid run is re-scored on **all 201 dev cards** with **the same rule as notebook 04** (dev top-3, then top-1, then dev loss). Only the noise is reduced; the rule does not change. The test set is not used.
2. **Are the differences real?** Notebook 05 reports point estimates on 201 (or 61) cases. Here we add paired bootstrap 95% confidence intervals and exact McNemar tests for the comparisons the report makes.
3. **Why do the 7–8B LLMs score only 0.30–0.38?** We split their errors into causes: names outside the list, diagnosing too early, the right answer at rank 2–3, and plain reasoning errors. We also check whether the errors come from a parsing bug in our code.

Outputs go to `data/processed/doctor/eval/` (records) and `results/doctor/` (tables and figures for the report).

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

MAX_QUESTIONS = 10          # same as notebook 05
SEED = 42
N_BOOT = 10_000
DATA_DIR = ROOT / "data" / "processed" / "doctor"
CKPT_DIR = ROOT / "checkpoints" / "doctor"
EVAL_DIR = DATA_DIR / "eval"
RESULTS = ROOT / "results" / "doctor"
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Setup
The same objects as notebook 05: the Symcat naive Bayes model, the lexicon, the disease-name matcher and the rule-based card patient.

In [2]:
import gc
import json
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from transformers import set_seed

from simclinic import cards as C
from simclinic.consult import diagnose_transcript, run_consultation, score
from simclinic.doctors import GPT2Doctor
from simclinic.lexicon import Lexicon, revealed_symptoms
from simclinic.naive_bayes import SymcatNaiveBayes
from simclinic.patients import CardPatient
from simclinic.prompts import DiseaseMatcher, parse_reply
from simclinic.symcat import Condition

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

CONDITIONS = json.loads((DATA_DIR / "conditions.json").read_text())
symcat_table = {n: Condition(**c) for n, c in json.loads((DATA_DIR / "symcat_conditions.json").read_text()).items()}
lexicon = Lexicon.load(symptoms=sorted({s for n in CONDITIONS for s in symcat_table[n].symptoms}))
nb = SymcatNaiveBayes({n: symcat_table[n].symptoms for n in CONDITIONS})
matcher = DiseaseMatcher({n: C.disease_names(symcat_table[n]) for n in CONDITIONS})

dev_cards = read_jsonl(DATA_DIR / "cards" / "dev.jsonl")
dev_dialogues = read_jsonl(DATA_DIR / "dialogues" / "dev.jsonl")
dev_seed = {c["card_id"]: SEED + i for i, c in enumerate(dev_cards)}
grid = pd.read_csv(CKPT_DIR / "grid_results.csv")
print(f"{len(dev_cards)} dev cards, {len(dev_dialogues)} dev dialogues, {len(grid)} grid runs")
grid

201 dev cards, 201 dev dialogues, 4 grid runs


,model,lr,epochs_run,best_epoch,dev_loss,train_minutes,dev_top1,dev_top3,dev_questions,dev_forced,path
0,gpt2,0.00005,5.0,5,0.660914,5.002747,0.52,0.84,6.32,0.0,checkpoints/doctor/gpt2-lr5e-05/best
1,gpt2,0.00010,5.0,5,0.630622,4.997283,0.64,0.92,6.10,0.0,checkpoints/doctor/gpt2-lr0.0001/best
2,gpt2-medium,0.00005,5.0,5,0.612245,13.121283,0.60,0.88,6.44,0.0,checkpoints/doctor/gpt2-medium-lr5e-05/best
3,gpt2-medium,0.00010,5.0,4,0.614604,13.079088,0.70,0.88,6.66,0.0,checkpoints/doctor/gpt2-medium-lr0.0001/best


## 2. Re-score every grid run on the full dev set
Each run interviews the rule-based patient for all 201 dev cards, with the same patient seed per card for every run, and also diagnoses the 201 held-out dev transcripts. Records are cached, so a rerun only does what is missing.

In [3]:
def slug(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def run_dev(run_name, path):
    auto_path = EVAL_DIR / "dev_auto" / f"{slug(run_name)}.jsonl"
    tr_path = EVAL_DIR / "dev_transcripts" / f"{slug(run_name)}.jsonl"
    for p in (auto_path, tr_path):
        p.parent.mkdir(parents=True, exist_ok=True)
    done_auto = {r["card_id"] for r in read_jsonl(auto_path)}
    done_tr = {r["dialogue_id"] for r in read_jsonl(tr_path)}
    todo_auto = [c for c in dev_cards if c["card_id"] not in done_auto]
    todo_tr = [d for d in dev_dialogues if d["dialogue_id"] not in done_tr]
    if todo_auto or todo_tr:
        t0 = time.time()
        doctor = GPT2Doctor.from_pretrained(str(ROOT / path), CONDITIONS)
        with open(auto_path, "a", encoding="utf-8") as f:
            for card in todo_auto:
                set_seed(dev_seed[card["card_id"]])
                rec = run_consultation(doctor, CardPatient(card, lexicon, seed=dev_seed[card["card_id"]]),
                                       max_questions=MAX_QUESTIONS)
                f.write(json.dumps({"run": run_name, "card_id": card["card_id"], "disease": card["disease"],
                                    **score(rec, card, nb, lexicon), **rec}, ensure_ascii=False) + "\n")
        with open(tr_path, "a", encoding="utf-8") as f:
            for d in todo_tr:
                set_seed(SEED)
                dx = diagnose_transcript(doctor, d["turns"])
                f.write(json.dumps({"run": run_name, "dialogue_id": d["dialogue_id"], "disease": d["disease"],
                                    "diagnosis": dx, "top1": dx[:1] == [d["disease"]],
                                    "top3": d["disease"] in dx[:3]}) + "\n")
        del doctor
        gc.collect()
        torch.cuda.empty_cache()
        print(f"{run_name}: {(time.time() - t0) / 60:.1f} min")
    return read_jsonl(auto_path), read_jsonl(tr_path)

dev_rows = []
for _, g in grid.iterrows():
    run_name = Path(g["path"]).parent.name
    auto, tr = run_dev(run_name, g["path"])
    dev_rows.append({"run": run_name, "model": g["model"], "lr": f'{g["lr"]:g}', "dev_loss": g["dev_loss"],
                     "dev50_top1": g["dev_top1"], "dev50_top3": g["dev_top3"],
                     "dev201_top1": np.mean([r["top1"] for r in auto]), "dev201_top3": np.mean([r["top3"] for r in auto]),
                     "dev201_questions": np.mean([r["n_questions"] for r in auto]),
                     "dev_transcript_top1": np.mean([r["top1"] for r in tr]),
                     "dev_transcript_top3": np.mean([r["top3"] for r in tr]), "path": g["path"]})
dev_table = pd.DataFrame(dev_rows)
dev_table.round(3)

,run,model,lr,dev_loss,dev50_top1,dev50_top3,dev201_top1,dev201_top3,dev201_questions,dev_transcript_top1,dev_transcript_top3,path
0,gpt2-lr5e-05,gpt2,5e-05,0.661,0.52,0.84,0.587,0.846,6.209,0.726,0.935,checkpoints/doctor/gpt2-lr5e-05/best
1,gpt2-lr0.0001,gpt2,0.0001,0.631,0.64,0.92,0.692,0.891,6.279,0.791,0.980,checkpoints/doctor/gpt2-lr0.0001/best
2,gpt2-medium-lr5e-05,gpt2-medium,5e-05,0.612,0.60,0.88,0.687,0.915,6.413,0.806,0.985,checkpoints/doctor/gpt2-medium-lr5e-05/best
3,gpt2-medium-lr0.0001,gpt2-medium,0.0001,0.615,0.70,0.88,0.731,0.930,6.328,0.856,0.995,checkpoints/doctor/gpt2-medium-lr0.0001/best


### 2.1 Selection
The rule is notebook 04's: highest dev auto-mode top-3, then top-1, then lowest dev loss. Only the number of dev cases changes (50 → 201).

In [4]:
ranked = dev_table.sort_values(["dev201_top3", "dev201_top1", "dev_loss"], ascending=[False, False, True])
best_per_size = ranked.groupby("model", sort=False).head(1).set_index("model")
selected = ranked.iloc[0]
print("selected on 201 dev cards:", selected["run"])
print("notebook 04's choice on 50 dev cards:", json.loads((CKPT_DIR / "best" / "doctor_config.json").read_text())["base_model"])
dev_table.round(3).to_csv(RESULTS / "dev_selection_201.csv", index=False)

# Paired comparison of the best small and best medium run on the dev cards, so the margin has an error bar.
def by_case(rows, key="card_id"):
    return {r[key]: r for r in rows}

a = by_case(read_jsonl(EVAL_DIR / "dev_auto" / f"{slug(Path(best_per_size.loc['gpt2-medium', 'path']).parent.name)}.jsonl"))
b = by_case(read_jsonl(EVAL_DIR / "dev_auto" / f"{slug(Path(best_per_size.loc['gpt2', 'path']).parent.name)}.jsonl"))
ids = sorted(set(a) & set(b))
for metric in ["top1", "top3"]:
    x = np.array([a[i][metric] for i in ids], float)
    y = np.array([b[i][metric] for i in ids], float)
    print(f"dev {metric}: medium {x.mean():.3f}, small {y.mean():.3f}, medium wins {int((x > y).sum())} cases, "
          f"small wins {int((y > x).sum())} cases")

selected on 201 dev cards: gpt2-medium-lr0.0001
notebook 04's choice on 50 dev cards: gpt2-medium
dev top1: medium 0.731, small 0.692, medium wins 26 cases, small wins 18 cases
dev top3: medium 0.930, small 0.891, medium wins 11 cases, small wins 3 cases


### 2.2 Update `checkpoints/doctor/best`
The app loads `checkpoints/doctor/best`. When the full-dev selection differs from notebook 04's, the selected run's weights and config are copied there. The old choice stays as `best_dev50/` so the change can be undone.

In [5]:
import shutil

best_dir = CKPT_DIR / "best"
config = json.loads((best_dir / "doctor_config.json").read_text())
sel_path = ROOT / selected["path"]
if config["base_model"] != selected["model"] or config["learning_rate"] != selected["lr"]:
    backup = CKPT_DIR / "best_dev50"
    if not backup.exists():
        shutil.copytree(best_dir, backup)
    for f in sel_path.iterdir():
        if f.is_file():
            shutil.copy2(f, best_dir / f.name)
    config.update({"base_model": selected["model"], "learning_rate": float(selected["lr"]),
                   "selected_on": "auto-mode dev top-3, then top-1, then dev loss; 201 dev cards (notebook 09)",
                   "scores": {"dev_loss": float(selected["dev_loss"]), "dev_top1": float(selected["dev201_top1"]),
                              "dev_top3": float(selected["dev201_top3"]),
                              "dev_questions": float(selected["dev201_questions"])}})
    (best_dir / "doctor_config.json").write_text(json.dumps(config, indent=2))
    print("checkpoints/doctor/best now holds", selected["run"], f"(previous choice kept in {backup.name}/)")
else:
    print("full-dev selection agrees with notebook 04; checkpoints/doctor/best unchanged")

checkpoints/doctor/best now holds gpt2-medium-lr0.0001 (previous choice kept in best_dev50/)


## 3. Test results with confidence intervals
Notebook 05's cached test records, re-scored with **paired bootstrap 95% intervals** (10,000 resamples of cases) and **exact McNemar tests** on the cases where two doctors disagree. All doctors in a table ran on the same cards with the same patient seed, so the comparisons are paired.

In [6]:
def load_eval(kind):
    rows = []
    for p in sorted((EVAL_DIR / kind).glob("*.jsonl")):
        rows += read_jsonl(p)
    return pd.DataFrame(rows)

auto = load_eval("auto")
trans = load_eval("transcripts")
llm_names = sorted(d for d in auto["doctor"].unique() if d.startswith(("llama", "mistral")))
llm_ids = set(auto[auto["doctor"].isin(llm_names)]["card_id"])
rng = np.random.default_rng(SEED)

def boot_ci(values, n_boot=N_BOOT):
    values = np.asarray(values, float)
    idx = rng.integers(0, len(values), (n_boot, len(values)))
    means = values[idx].mean(1)
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def ci_table(df, key, metrics=("top1", "top3")):
    out = []
    for doctor, g in df.groupby("doctor", sort=False):
        row = {"doctor": doctor, "cases": len(g)}
        for m in metrics:
            mean, lo, hi = boot_ci(g[m])
            row[m] = f"{mean:.3f} [{lo:.3f}, {hi:.3f}]"
        out.append(row)
    return pd.DataFrame(out).set_index("doctor")

ORDER = ["NB doctor", "GPT-2 pretrained (prompt only)", "GPT-2 small fine-tuned", "GPT-2 medium fine-tuned"] + llm_names
def ordered(df):
    return df.assign(_o=df["doctor"].map({d: i for i, d in enumerate(ORDER)}).fillna(99)).sort_values("_o").drop(columns="_o")

auto_all = ordered(auto[auto["doctor"].isin(ORDER[:4])])
auto_ci = ci_table(auto_all, "card_id")
auto_ci_llm = ci_table(ordered(auto[auto["card_id"].isin(llm_ids)]), "card_id")
trans_ci = ci_table(ordered(trans), "dialogue_id")
auto_ci.to_csv(RESULTS / "test_auto_ci_201.csv")
auto_ci_llm.to_csv(RESULTS / "test_auto_ci_61.csv")
print("Auto mode, 201 test cards"); display(auto_ci)
print("Auto mode, 61-card LLM subset"); display(auto_ci_llm)

Auto mode, 201 test cards


,cases,top1,top3
doctor,,,
NB doctor,201,"0.706 [0.642, 0.766]","0.940 [0.905, 0.970]"
GPT-2 pretrained (prompt only),201,"0.050 [0.020, 0.085]","0.080 [0.045, 0.119]"
GPT-2 small fine-tuned,201,"0.687 [0.622, 0.751]","0.861 [0.811, 0.905]"
GPT-2 medium fine-tuned,201,"0.726 [0.662, 0.786]","0.900 [0.856, 0.940]"


Auto mode, 61-card LLM subset


,cases,top1,top3
doctor,,,
NB doctor,61,"0.738 [0.623, 0.836]","0.951 [0.885, 1.000]"
GPT-2 pretrained (prompt only),61,"0.033 [0.000, 0.082]","0.082 [0.016, 0.148]"
GPT-2 small fine-tuned,61,"0.672 [0.557, 0.787]","0.885 [0.803, 0.967]"
GPT-2 medium fine-tuned,61,"0.770 [0.672, 0.869]","0.918 [0.836, 0.984]"
llama3-8b cot,61,"0.328 [0.213, 0.443]","0.557 [0.426, 0.672]"
llama3-8b few_shot,61,"0.295 [0.180, 0.410]","0.574 [0.443, 0.705]"
llama3-8b zero_shot,61,"0.377 [0.262, 0.492]","0.639 [0.525, 0.754]"
mistral-7b cot,61,"0.344 [0.230, 0.459]","0.492 [0.361, 0.607]"
mistral-7b few_shot,61,"0.361 [0.246, 0.475]","0.623 [0.508, 0.738]"


### 3.1 Paired significance tests
McNemar's exact test: among the cases where exactly one of two doctors is right, is the split far from 50/50?

In [7]:
def mcnemar(df, d1, d2, metric, key="card_id", setting="auto mode"):
    a = df[df["doctor"] == d1].set_index(key)[metric]
    b = df[df["doctor"] == d2].set_index(key)[metric]
    common = a.index.intersection(b.index)
    a, b = a[common].astype(bool), b[common].astype(bool)
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {"setting": setting, "comparison": f"{d1} vs {d2}", "metric": metric, "cases": len(common),
            f"acc A": round(a.mean(), 3), f"acc B": round(b.mean(), 3),
            "only A right": only_a, "only B right": only_b, "p (exact McNemar)": round(p, 4)}

best_llm = (auto[auto["doctor"].isin(llm_names)].groupby("doctor")["top1"].mean().idxmax())
tests = []
for metric in ["top1", "top3"]:
    tests += [mcnemar(auto, "GPT-2 medium fine-tuned", "GPT-2 small fine-tuned", metric),
              mcnemar(auto, "GPT-2 medium fine-tuned", "NB doctor", metric),
              mcnemar(auto, "GPT-2 small fine-tuned", "NB doctor", metric),
              mcnemar(auto[auto["card_id"].isin(llm_ids)], "GPT-2 medium fine-tuned", best_llm, metric),
              mcnemar(trans, "GPT-2 medium fine-tuned", "GPT-2 small fine-tuned", metric, key="dialogue_id", setting="transcripts")]
sig = pd.DataFrame(tests)
sig.to_csv(RESULTS / "test_mcnemar.csv", index=False)
sig

,setting,comparison,metric,cases,acc A,acc B,only A right,only B right,p (exact McNemar)
0,auto mode,GPT-2 medium fine-tuned vs GPT-2 small fine-tuned,top1,201,0.726,0.687,27,19,0.3020
1,auto mode,GPT-2 medium fine-tuned vs NB doctor,top1,201,0.726,0.706,32,28,0.6989
2,auto mode,GPT-2 small fine-tuned vs NB doctor,top1,201,0.687,0.706,26,30,0.6889
3,auto mode,GPT-2 medium fine-tuned vs llama3-8b zero_shot,top1,61,0.770,0.377,28,4,0.0000
4,transcripts,GPT-2 medium fine-tuned vs GPT-2 small fine-tuned,top1,201,0.856,0.816,18,10,0.1849
5,auto mode,GPT-2 medium fine-tuned vs GPT-2 small fine-tuned,top3,201,0.900,0.861,15,7,0.1338
6,auto mode,GPT-2 medium fine-tuned vs NB doctor,top3,201,0.900,0.940,6,14,0.1153
7,auto mode,GPT-2 small fine-tuned vs NB doctor,top3,201,0.861,0.940,5,21,0.0025
8,auto mode,GPT-2 medium fine-tuned vs llama3-8b zero_shot,top3,61,0.918,0.639,18,1,0.0001
9,transcripts,GPT-2 medium fine-tuned vs GPT-2 small fine-tuned,top3,201,0.975,0.990,1,4,0.3750


Notebook 05 computes the transcript naive Bayes on the fly instead of caching it. It is re-created here so the GPT-2 vs NB transcript test above can use it.

In [8]:
if "Naive Bayes" not in set(trans["doctor"]):
    test_dialogues = read_jsonl(DATA_DIR / "dialogues" / "test.jsonl")
    nb_rows = []
    for d in test_dialogues:
        dx = [x for x, _ in nb.rank(revealed_symptoms(d["turns"][:-1], lexicon), k=3)]
        nb_rows.append({"doctor": "Naive Bayes", "dialogue_id": d["dialogue_id"], "disease": d["disease"],
                        "diagnosis": dx, "top1": dx[:1] == [d["disease"]], "top3": d["disease"] in dx})
    trans = pd.concat([trans, pd.DataFrame(nb_rows)], ignore_index=True)
extra = pd.DataFrame([mcnemar(trans, "GPT-2 medium fine-tuned", "Naive Bayes", m, key="dialogue_id", setting="transcripts") for m in ["top1", "top3"]])
sig = pd.concat([sig, extra], ignore_index=True)
sig.to_csv(RESULTS / "test_mcnemar.csv", index=False)
trans_ci = ci_table(ordered(trans), "dialogue_id")
trans_ci.to_csv(RESULTS / "test_transcripts_ci.csv")
display(extra)
print("Fixed transcripts (201 test dialogues; LLMs: 61)"); display(trans_ci)

,setting,comparison,metric,cases,acc A,acc B,only A right,only B right,p (exact McNemar)
0,transcripts,GPT-2 medium fine-tuned vs Naive Bayes,top1,201,0.856,0.642,56,13,0.0000
1,transcripts,GPT-2 medium fine-tuned vs Naive Bayes,top3,201,0.975,0.910,17,4,0.0072


Fixed transcripts (201 test dialogues; LLMs: 61)


,cases,top1,top3
doctor,,,
GPT-2 pretrained (prompt only),201,"0.050 [0.020, 0.080]","0.090 [0.055, 0.129]"
GPT-2 small fine-tuned,201,"0.816 [0.761, 0.866]","0.990 [0.975, 1.000]"
GPT-2 medium fine-tuned,201,"0.856 [0.806, 0.900]","0.975 [0.950, 0.995]"
llama3-8b cot,61,"0.328 [0.213, 0.443]","0.541 [0.410, 0.672]"
llama3-8b few_shot,61,"0.361 [0.246, 0.475]","0.689 [0.574, 0.803]"
llama3-8b zero_shot,61,"0.410 [0.295, 0.541]","0.672 [0.557, 0.787]"
mistral-7b cot,61,"0.377 [0.262, 0.508]","0.574 [0.443, 0.689]"
mistral-7b few_shot,61,"0.393 [0.279, 0.525]","0.623 [0.492, 0.738]"
mistral-7b zero_shot,61,"0.361 [0.246, 0.475]","0.639 [0.525, 0.754]"


## 4. Per-disease accuracy
Which conditions does the shipped doctor get wrong? There are 3–4 test cards per disease, so single diseases are noisy. We group the 61 conditions by body system, which gives groups of 12–40 cases.

In [9]:
SYSTEMS = {
    "ENT / respiratory": ["Flu", "Common cold", "Strep throat", "Acute sinusitis", "Acute otitis media",
                          "Otitis externa (swimmer's ear)", "Tonsillitis", "Laryngitis", "Acute bronchitis",
                          "Pneumonia", "Asthma", "Seasonal allergies (hay fever)", "Ear wax impaction",
                          "Mononucleosis", "Herpangina"],
    "Eye": ["Conjunctivitis", "Stye"],
    "Digestive": ["Infectious gastroenteritis", "Gastroesophageal reflux disease (GERD)", "Gastritis", "Indigestion",
                  "Irritable bowel syndrome", "Chronic constipation", "Hemorrhoids", "Appendicitis", "Gallstone",
                  "Lactose intolerance", "Aphthous ulcer"],
    "Urogenital": ["Urinary tract infection", "Kidney stone", "Pyelonephritis", "Vaginal yeast infection",
                   "Benign prostatic hyperplasia (BPH)"],
    "Musculoskeletal": ["Lumbago", "Sprain or strain", "Gout", "Osteoarthritis", "Carpal tunnel syndrome",
                        "Muscle spasm", "Sciatica", "Lateral epicondylitis (tennis elbow)"],
    "Neurological": ["Migraine", "Tension headache", "Benign paroxysmal positional vertigo (BPPV)"],
    "Skin": ["Eczema", "Contact dermatitis", "Fungal infection of the skin", "Chickenpox", "Shingles (herpes zoster)",
             "Impetigo", "Cold sore", "Insect bite", "Athlete's foot", "Viral exanthem",
             "Hand, foot and mouth disease (HFMD)"],
    "Mental health / sleep": ["Anxiety", "Depression", "Primary insomnia", "Panic attack"],
    "Systemic": ["Heat exhaustion", "Dengue fever"],
}
system_of = {d: s for s, ds in SYSTEMS.items() for d in ds}
assert set(system_of) == set(CONDITIONS), set(CONDITIONS) ^ set(system_of)
shown = ["NB doctor", "GPT-2 small fine-tuned", "GPT-2 medium fine-tuned"]
per_system = (auto_all[auto_all["doctor"].isin(shown)].assign(system=lambda d: d["disease"].map(system_of))
              .pivot_table(index="system", columns="doctor", values="top1", aggfunc="mean")[shown])
per_system.insert(0, "cases", auto_all[auto_all["doctor"] == shown[0]]["disease"].map(system_of).value_counts())
per_system.round(3).to_csv(RESULTS / "test_auto_by_system.csv")
per_system.round(3)

doctor,cases,NB doctor,GPT-2 small fine-tuned,GPT-2 medium fine-tuned
system,,,,
Digestive,37,0.676,0.757,0.892
ENT / respiratory,52,0.558,0.615,0.615
Eye,7,1.000,1.000,1.000
Mental health / sleep,12,0.917,1.000,0.917
Musculoskeletal,24,0.875,0.917,1.000
Neurological,10,0.900,0.800,0.700
Skin,36,0.528,0.278,0.306
Systemic,6,0.833,0.667,0.833
Urogenital,17,0.941,0.882,0.941


In [10]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.2, 3.6))
plot = per_system.drop(columns="cases").sort_values("GPT-2 medium fine-tuned")
y = np.arange(len(plot))
colors = {"NB doctor": "#9aa3ad", "GPT-2 small fine-tuned": "#7fb3d5", "GPT-2 medium fine-tuned": "#1f5f99"}
for i, d in enumerate(shown):
    ax.barh(y + (i - 1) * 0.27, plot[d], height=0.26, color=colors[d], label=d.replace(" fine-tuned", ""))
ax.set_yticks(y, [f"{s} (n={per_system.loc[s, 'cases']})" for s in plot.index], fontsize=8)
ax.set_xlim(0, 1)
ax.set_xlabel("Top-1 accuracy, auto mode (test)")
ax.legend(fontsize=8, frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS / "fig_accuracy_by_system.pdf")
fig.savefig(RESULTS / "fig_accuracy_by_system.png", dpi=200)

## 5. Why the LLM doctors score low
Each wrong LLM top-1 on the 61-case subset gets exactly one cause, checked in this order:

| cause | test |
|---|---|
| **no valid name** | the parsed differential is empty |
| **out-of-list name first** | the raw DIAGNOSIS line starts with a name the matcher maps to no candidate (e.g. "Pharyngitis") |
| **too few questions** | the doctor stopped after ≤ 3 questions and naive Bayes on its evidence is also wrong |
| **right answer at rank 2–3** | the true disease is in the differential but not first |
| **evidence was enough** | naive Bayes on the same revealed symptoms gets it right, but the LLM does not |
| **other** | none of the above |

Before blaming the models, we check our parser. For every LLM reply, the true disease is searched in the *raw* text. If it appears in the DIAGNOSIS line but was not parsed, that is our bug, not the model's.

In [11]:
llm = auto[auto["doctor"].isin(llm_names)].copy()

def dx_line(raw):
    raw = (raw or "").split("\n[retry]\n")[-1]
    m = list(re.finditer(r"DIAGNOSIS\s*:\s*(.+)", re.sub(r"[*_`#]", "", raw), re.I))
    return m[-1].group(1) if m else ""

def first_item(line):
    items = [x.strip(" .,-[]") for x in re.split(r"\s*(?:;|\n|\b\d+[.)]\s)\s*", line) if x.strip(" .,-[]")]
    if len(items) == 1 and "," in items[0]:
        items = [x.strip() for x in items[0].split(",")]
    return items[0] if items else ""

def cause(r):
    if r["top1"]:
        return "correct"
    if not r["diagnosis"]:
        return "no valid name"
    first = first_item(dx_line(r["raw"]))
    if first and matcher(first) is None:
        return "out-of-list name first"
    if r["n_questions"] <= 3 and not r["nb_top1"]:
        return "too few questions"
    if r["top3"]:
        return "right answer at rank 2-3"
    if r["nb_top1"]:
        return "evidence was enough"
    return "other"

llm["cause"] = llm.apply(cause, axis=1)
causes = pd.crosstab(llm["doctor"], llm["cause"], normalize="index").round(3)
causes = causes[[c for c in ["correct", "no valid name", "out-of-list name first", "too few questions",
                             "right answer at rank 2-3", "evidence was enough", "other"] if c in causes]]
causes.to_csv(RESULTS / "llm_error_causes.csv")
causes

cause,correct,no valid name,out-of-list name first,too few questions,right answer at rank 2-3,evidence was enough,other
doctor,,,,,,,
llama3-8b cot,0.328,0.000,0.016,0.000,0.230,0.197,0.230
llama3-8b few_shot,0.295,0.000,0.016,0.000,0.279,0.230,0.180
llama3-8b zero_shot,0.377,0.000,0.000,0.180,0.213,0.180,0.049
mistral-7b cot,0.344,0.016,0.082,0.000,0.115,0.279,0.164
mistral-7b few_shot,0.361,0.000,0.049,0.000,0.246,0.164,0.180
mistral-7b zero_shot,0.295,0.000,0.049,0.016,0.180,0.230,0.230


In [12]:
def parser_missed(r):
    """True disease written in the raw DIAGNOSIS line, but not first in the parsed differential."""
    line = dx_line(r["raw"])
    names = [n for n in C.disease_names(symcat_table[r["disease"]]) if len(n) >= 4] if r["disease"] in symcat_table else [r["disease"]]
    said = any(re.search(rf"\b{re.escape(n.lower())}\b", line.lower()) for n in names)
    return said and line and matcher(first_item(line)) == r["disease"] and not r["top1"]

missed = llm[llm.apply(parser_missed, axis=1)]
print(f"replies where the parser lost a correct first answer: {len(missed)} of {len(llm)}")
out_of_list = Counter(first_item(dx_line(r)) for r in llm[llm["cause"] == "out-of-list name first"]["raw"])
print("most common out-of-list first names:", out_of_list.most_common(12))

replies where the parser lost a correct first answer: 0 of 366
most common out-of-list first names: [('Allergies', 2), ("Based on the patient's symptoms", 1), ('DIAGNOSIS: Flu', 1), ('Otitis media with effusion', 1), ('Abdominal pain without clear cause', 1), ('Acute vestibular neuronitis', 1), ('Allergic reaction', 1), ('Psoriasis', 1), ('Chronic sinusitis', 1), ('Sinusitis', 1), ('Fungal infection', 1), ('Nausea with fever and headache', 1)]


### 5.1 Do the LLMs favour a few common diagnoses?
A model that falls back on familiar names concentrates its predictions. We count distinct first choices and the share taken by the five most frequent ones, against the 61 different true diseases.

In [13]:
conc = []
for d, g in auto[auto["card_id"].isin(llm_ids)].groupby("doctor", sort=False):
    firsts = [x[0] for x in g["diagnosis"] if x]
    c = Counter(firsts)
    conc.append({"doctor": d, "distinct first choices": len(c), "top-5 share": round(sum(n for _, n in c.most_common(5)) / max(1, len(firsts)), 3),
                 "most common": ", ".join(f"{k} ({v})" for k, v in c.most_common(3))})
conc = ordered(pd.DataFrame(conc)).set_index("doctor")
conc.to_csv(RESULTS / "prediction_concentration.csv")
conc

,distinct first choices,top-5 share,most common
doctor,,,
NB doctor,48,0.164,"Mononucleosis (2), Seasonal allergies (hay fever) (2), Common cold (2)"
GPT-2 pretrained (prompt only),6,0.984,"Pneumonia (45), Depression (8), Osteoarthritis (4)"
GPT-2 small fine-tuned,46,0.262,"Viral exanthem (5), Mononucleosis (3), Irritable bowel syndrome (3)"
GPT-2 medium fine-tuned,49,0.246,"Insect bite (4), Chickenpox (4), Viral exanthem (3)"
llama3-8b cot,33,0.361,"Infectious gastroenteritis (6), Common cold (5), Lumbago (4)"
llama3-8b few_shot,26,0.459,"Chickenpox (7), Common cold (6), Lumbago (6)"
llama3-8b zero_shot,30,0.393,"Lumbago (6), Strep throat (5), Chickenpox (5)"
mistral-7b cot,37,0.300,"Infectious gastroenteritis (6), Tonsillitis (3), Lumbago (3)"
mistral-7b few_shot,30,0.410,"Infectious gastroenteritis (8), Lumbago (5), Anxiety (5)"


## 6. Findings
Written after the run (section 7 of notebook 05 has the point estimates). See the cell below for the numbers each statement uses.

In [14]:
print(dev_table[["run", "dev50_top1", "dev50_top3", "dev201_top1", "dev201_top3", "dev_transcript_top1"]].round(3).to_string(index=False))
print()
print(sig.to_string(index=False))

                 run  dev50_top1  dev50_top3  dev201_top1  dev201_top3  dev_transcript_top1
        gpt2-lr5e-05        0.52        0.84        0.587        0.846                0.726
       gpt2-lr0.0001        0.64        0.92        0.692        0.891                0.791
 gpt2-medium-lr5e-05        0.60        0.88        0.687        0.915                0.806
gpt2-medium-lr0.0001        0.70        0.88        0.731        0.930                0.856

    setting                                        comparison metric  cases  acc A  acc B  only A right  only B right  p (exact McNemar)
  auto mode GPT-2 medium fine-tuned vs GPT-2 small fine-tuned   top1    201  0.726  0.687            27            19             0.3020
  auto mode              GPT-2 medium fine-tuned vs NB doctor   top1    201  0.726  0.706            32            28             0.6989
  auto mode               GPT-2 small fine-tuned vs NB doctor   top1    201  0.687  0.706            26            30           